# W1 Day1 (04/07 周一): Python 核心 + 并发

## 学习目标
- **理论 (1-1.5h)**: 对象模型 / 引用计数 / GC 三代 / `__slots__`; GIL / threading / multiprocessing / asyncio
- **实践 (2h)**: GIL 多线程 vs 多进程 benchmark + asyncio 爬虫 demo
- **产出物**: 并发对比 notebook

---

## 目录
1. [Python 对象模型](#1)
2. [引用计数 + GC 三代](#2)
3. [`__slots__` 优化](#3)
4. [GIL 深入理解](#4)
5. [threading 多线程](#5)
6. [multiprocessing 多进程](#6)
7. [asyncio 异步编程](#7)
8. [并发模型大 Benchmark](#8)
9. [asyncio 爬虫 Demo](#9)
10. [思考题 & 总结](#10)

---
## 1. Python 对象模型 <a id='1'></a>

### 1.1 一切皆对象

Python 中**所有东西**都是对象：整数、字符串、函数、类本身。
每个对象都有三个要素:
- **id**: 内存地址 (CPython 中就是 C 指针)
- **type**: 类型 (也是一个对象)
- **value**: 值

### 1.2 变量是标签，不是盒子

Python 变量本质上是**引用 (reference)**，指向对象，而非包含值的容器。

In [1]:
import sys
import gc
import ctypes
import weakref
import time
import os

# 1.1 一切皆对象
print("=" * 60)
print("Python 对象模型")
print("=" * 60)

x = 42
print(f"x = 42")
print(f"  id(x)   = {id(x)}")
print(f"  type(x) = {type(x)}")
print(f"  x.__class__.__mro__ = {x.__class__.__mro__}")
print()

# 函数也是对象
def foo(): pass
print(f"函数也是对象: type(foo) = {type(foo)}")
print(f"  foo 的属性: {[a for a in dir(foo) if not a.startswith('__')][:5]}...")
print()

# type 本身也是对象
print(f"type 的类型: type(type) = {type(type)}")
print(f"type 是 object 的子类: {issubclass(type, object)}")
print(f"object 的类型是 type: {type(object) is type}")
print("\n💡 type 和 object 互相定义 —— 这是 Python 的 bootstrapping 鸡生蛋问题")

Python 对象模型
x = 42
  id(x)   = 140725366405320
  type(x) = <class 'int'>
  x.__class__.__mro__ = (<class 'int'>, <class 'object'>)

函数也是对象: type(foo) = <class 'function'>
  foo 的属性: []...

type 的类型: type(type) = <class 'type'>
type 是 object 的子类: True
object 的类型是 type: True

💡 type 和 object 互相定义 —— 这是 Python 的 bootstrapping 鸡生蛋问题


In [2]:
# 1.2 变量是引用
print("\n" + "=" * 60)
print("变量是引用，不是盒子")
print("=" * 60)

a = [1, 2, 3]
b = a  # b 指向同一个对象
print(f"a = [1, 2, 3]; b = a")
print(f"  id(a) == id(b): {id(a) == id(b)}")
print(f"  a is b: {a is b}")

b.append(4)
print(f"\nb.append(4) 后:")
print(f"  a = {a}  ← a 也变了！")
print(f"  b = {b}")

# 小整数缓存 (interning)
print("\n--- 小整数缓存 ---")
x = 256
y = 256
print(f"x = 256; y = 256; x is y: {x is y}  ← [-5, 256] 范围内的整数被缓存")

x = 257
y = 257
# 注意: 在交互式环境和脚本中行为可能不同
print(f"x = 257; y = 257; x is y: {x is y}  ← 超出缓存范围")

# 字符串驻留
s1 = "hello"
s2 = "hello"
print(f"\ns1 = 'hello'; s2 = 'hello'; s1 is s2: {s1 is s2}  ← 字符串驻留")


变量是引用，不是盒子
a = [1, 2, 3]; b = a
  id(a) == id(b): True
  a is b: True

b.append(4) 后:
  a = [1, 2, 3, 4]  ← a 也变了！
  b = [1, 2, 3, 4]

--- 小整数缓存 ---
x = 256; y = 256; x is y: True  ← [-5, 256] 范围内的整数被缓存
x = 257; y = 257; x is y: False  ← 超出缓存范围

s1 = 'hello'; s2 = 'hello'; s1 is s2: True  ← 字符串驻留


In [3]:
# 1.3 对象内存布局
print("\n" + "=" * 60)
print("对象内存占用 (sys.getsizeof)")
print("=" * 60)

objects = [
    ("int(0)", 0),
    ("int(1)", 1),
    ("int(2**30)", 2**30),
    ("int(2**100)", 2**100),
    ("float(3.14)", 3.14),
    ("str('')", ""),
    ("str('hello')", "hello"),
    ("list([])", []),
    ("list([1,2,3])", [1, 2, 3]),
    ("dict({})", {}),
    ("dict({1:1})", {1: 1}),
    ("set()", set()),
    ("tuple(())", ()),
    ("None", None),
    ("True", True),
]

for name, obj in objects:
    print(f"  {name:20s}: {sys.getsizeof(obj):>4d} bytes")

print("\n💡 即使空 dict 也占 64 bytes，因为有哈希表的 overhead")
print("💡 大整数 (>2^30) 会动态分配更多内存 (变长整数)")


对象内存占用 (sys.getsizeof)
  int(0)              :   28 bytes
  int(1)              :   28 bytes
  int(2**30)          :   32 bytes
  int(2**100)         :   40 bytes
  float(3.14)         :   24 bytes
  str('')             :   41 bytes
  str('hello')        :   46 bytes
  list([])            :   56 bytes
  list([1,2,3])       :   88 bytes
  dict({})            :   64 bytes
  dict({1:1})         :  224 bytes
  set()               :  216 bytes
  tuple(())           :   40 bytes
  None                :   16 bytes
  True                :   28 bytes

💡 即使空 dict 也占 64 bytes，因为有哈希表的 overhead
💡 大整数 (>2^30) 会动态分配更多内存 (变长整数)


---
## 2. 引用计数 + GC 三代 <a id='2'></a>

### 2.1 CPython 的内存管理: 双层机制

1. **引用计数 (Reference Counting)**: 主要机制，实时回收
   - 每个对象维护一个引用计数器
   - 计数变为 0 时立即释放
   - 优点: 实时、确定性
   - 缺点: 无法处理循环引用

2. **分代垃圾回收 (Generational GC)**: 补充机制，处理循环引用
   - 3 代: Gen0 (新生) → Gen1 (中年) → Gen2 (老年)
   - 新对象进入 Gen0，存活越久升级到更高代
   - 高代扫描频率低 ("弱分代假说": 大部分对象朝生暮死)

In [4]:
# 2.1 引用计数实验
print("=" * 60)
print("引用计数实验")
print("=" * 60)

class Traced:
    """带析构追踪的类"""
    def __init__(self, name):
        self.name = name
        print(f"  [CREATE] {name} (id={id(self)})")
    
    def __del__(self):
        print(f"  [DELETE] {self.name}")

print("\n--- 正常引用计数 ---")
a = Traced("obj_A")
print(f"  refcount(a) = {sys.getrefcount(a) - 1}")  # -1 因为 getrefcount 本身创建一个临时引用

b = a  # 增加引用
print(f"  b = a → refcount = {sys.getrefcount(a) - 1}")

c = [a]  # 列表也持有引用
print(f"  c = [a] → refcount = {sys.getrefcount(a) - 1}")

del b
print(f"  del b → refcount = {sys.getrefcount(a) - 1}")

del c
print(f"  del c → refcount = {sys.getrefcount(a) - 1}")

print("  del a →", end=" ")
del a  # 引用计数变为0，立即触发 __del__

引用计数实验

--- 正常引用计数 ---
  [CREATE] obj_A (id=2242961149632)
  refcount(a) = 1
  b = a → refcount = 2
  c = [a] → refcount = 3
  del b → refcount = 2
  del c → refcount = 1
  del a →   [DELETE] obj_A


In [5]:
# 2.2 循环引用问题
print("\n" + "=" * 60)
print("循环引用 + GC 实验")
print("=" * 60)

gc.disable()  # 暂时关闭 GC 以观察

class Node:
    def __init__(self, name):
        self.name = name
        self.partner = None
    def __repr__(self):
        return f"Node({self.name})"
    def __del__(self):
        print(f"  [GC collected] Node({self.name})")

# 创建循环引用
print("\n创建循环引用: A ↔ B")
nodeA = Node("A")
nodeB = Node("B")
nodeA.partner = nodeB
nodeB.partner = nodeA

# 保存 id 用于后续检查
id_a = id(nodeA)
id_b = id(nodeB)

print(f"  refcount(A) = {sys.getrefcount(nodeA) - 1}")
print(f"  refcount(B) = {sys.getrefcount(nodeB) - 1}")

# 删除外部引用
del nodeA
del nodeB
print("\ndel nodeA, nodeB 后:")
print("  对象没有被释放！(引用计数仍为1，因为循环引用)")

# 手动触发 GC
print("\n手动 gc.collect():")
collected = gc.collect()
print(f"  收集了 {collected} 个对象")

gc.enable()  # 重新启用


循环引用 + GC 实验

创建循环引用: A ↔ B
  refcount(A) = 2
  refcount(B) = 2

del nodeA, nodeB 后:
  对象没有被释放！(引用计数仍为1，因为循环引用)

手动 gc.collect():
  [GC collected] Node(A)
  [GC collected] Node(B)
  收集了 2 个对象


In [6]:
# 2.3 GC 三代机制
print("\n" + "=" * 60)
print("GC 三代 (Generational GC)")
print("=" * 60)

# 查看各代的阈值
thresholds = gc.get_threshold()
print(f"各代回收阈值: Gen0={thresholds[0]}, Gen1={thresholds[1]}, Gen2={thresholds[2]}")
print(f"  含义: Gen0 每 {thresholds[0]} 次分配-释放差触发一次")
print(f"        Gen1 每 {thresholds[1]} 次 Gen0 收集触发一次")
print(f"        Gen2 每 {thresholds[2]} 次 Gen1 收集触发一次")

# 当前各代对象数
counts = gc.get_count()
print(f"\n当前各代对象数: Gen0={counts[0]}, Gen1={counts[1]}, Gen2={counts[2]}")

# 实验: 对象从 Gen0 晋升到 Gen1
print("\n--- 对象晋升实验 ---")
gc.collect()  # 清理
print(f"gc.collect() 后: {gc.get_count()}")

# 创建一些对象
objects_keep = [object() for _ in range(100)]
print(f"创建100个对象后: {gc.get_count()}")

gc.collect(0)  # 只收集 Gen0
print(f"gc.collect(0) 后 (Gen0→Gen1晋升): {gc.get_count()}")

gc.collect(1)  # 收集 Gen0 和 Gen1
print(f"gc.collect(1) 后 (Gen1→Gen2晋升): {gc.get_count()}")

del objects_keep


GC 三代 (Generational GC)
各代回收阈值: Gen0=2000, Gen1=10, Gen2=10
  含义: Gen0 每 2000 次分配-释放差触发一次
        Gen1 每 10 次 Gen0 收集触发一次
        Gen2 每 10 次 Gen1 收集触发一次

当前各代对象数: Gen0=848, Gen1=0, Gen2=0

--- 对象晋升实验 ---
gc.collect() 后: (25, 0, 0)
创建100个对象后: (30, 0, 0)
gc.collect(0) 后 (Gen0→Gen1晋升): (3, 1, 0)
gc.collect(1) 后 (Gen1→Gen2晋升): (3, 0, 1)


In [7]:
# 2.4 weakref: 避免循环引用的工具
print("\n" + "=" * 60)
print("weakref: 弱引用")
print("=" * 60)

class Resource:
    def __init__(self, name):
        self.name = name
    def __repr__(self):
        return f"Resource({self.name})"
    def __del__(self):
        print(f"  [释放] Resource({self.name})")

# 强引用 vs 弱引用
obj = Resource("data")
strong_ref = obj
weak_ref = weakref.ref(obj)

print(f"强引用: {strong_ref}")
print(f"弱引用: {weak_ref()}")
print(f"refcount: {sys.getrefcount(obj) - 1} (弱引用不增加计数!)")

del obj, strong_ref
print(f"\ndel obj, strong_ref 后:")
print(f"  weak_ref() = {weak_ref()}  ← 对象已被回收，弱引用返回 None")

print("\n💡 实际应用: 缓存 (weakref.WeakValueDictionary), 观察者模式, 避免循环引用")


weakref: 弱引用
强引用: Resource(data)
弱引用: Resource(data)
refcount: 2 (弱引用不增加计数!)
  [释放] Resource(data)

del obj, strong_ref 后:
  weak_ref() = None  ← 对象已被回收，弱引用返回 None

💡 实际应用: 缓存 (weakref.WeakValueDictionary), 观察者模式, 避免循环引用


---
## 3. `__slots__` 优化 <a id='3'></a>

默认情况下，每个实例用 `__dict__` 存储属性 → 灵活但占内存。

`__slots__` 声明固定属性集合 → 用固定偏移替代哈希表 → 省内存 + 略快。

In [8]:
import tracemalloc

class PointDict:
    """普通类 (使用 __dict__)"""
    def __init__(self, x, y, z):
        self.x = x
        self.y = y
        self.z = z

class PointSlots:
    """使用 __slots__ 优化"""
    __slots__ = ('x', 'y', 'z')
    
    def __init__(self, x, y, z):
        self.x = x
        self.y = y
        self.z = z

print("=" * 60)
print("__slots__ vs __dict__ 内存对比")
print("=" * 60)

# 单个实例内存
p_dict = PointDict(1.0, 2.0, 3.0)
p_slots = PointSlots(1.0, 2.0, 3.0)

print(f"\n单实例内存:")
print(f"  PointDict:  {sys.getsizeof(p_dict)} + __dict__({sys.getsizeof(p_dict.__dict__)}) "
      f"= {sys.getsizeof(p_dict) + sys.getsizeof(p_dict.__dict__)} bytes")
print(f"  PointSlots: {sys.getsizeof(p_slots)} bytes")
print(f"  节省: {sys.getsizeof(p_dict) + sys.getsizeof(p_dict.__dict__) - sys.getsizeof(p_slots)} bytes")

# 大规模对比
N = 100_000

tracemalloc.start()
dict_objects = [PointDict(i, i+1, i+2) for i in range(N)]
dict_mem = tracemalloc.get_traced_memory()[1]
tracemalloc.stop()
del dict_objects

tracemalloc.start()
slot_objects = [PointSlots(i, i+1, i+2) for i in range(N)]
slot_mem = tracemalloc.get_traced_memory()[1]
tracemalloc.stop()
del slot_objects

print(f"\n{N:,} 个实例的峰值内存:")
print(f"  PointDict:  {dict_mem / 1024 / 1024:.1f} MB")
print(f"  PointSlots: {slot_mem / 1024 / 1024:.1f} MB")
print(f"  节省: {(1 - slot_mem / dict_mem) * 100:.0f}%")

__slots__ vs __dict__ 内存对比

单实例内存:
  PointDict:  48 + __dict__(296) = 344 bytes
  PointSlots: 56 bytes
  节省: 288 bytes

100,000 个实例的峰值内存:
  PointDict:  19.2 MB
  PointSlots: 15.4 MB
  节省: 20%


In [9]:
# __slots__ 属性访问速度
print("\n--- __slots__ 属性访问速度 ---")

p_d = PointDict(1.0, 2.0, 3.0)
p_s = PointSlots(1.0, 2.0, 3.0)

n_iters = 1_000_000

start = time.perf_counter()
for _ in range(n_iters):
    _ = p_d.x + p_d.y + p_d.z
dict_time = time.perf_counter() - start

start = time.perf_counter()
for _ in range(n_iters):
    _ = p_s.x + p_s.y + p_s.z
slots_time = time.perf_counter() - start

print(f"  __dict__  读取 {n_iters:,} 次: {dict_time:.3f}s")
print(f"  __slots__ 读取 {n_iters:,} 次: {slots_time:.3f}s")
print(f"  加速比: {dict_time / slots_time:.2f}x")

# __slots__ 的限制
print("\n--- __slots__ 限制 ---")
try:
    p_s.w = 4.0  # 不能动态添加属性
except AttributeError as e:
    print(f"  不能动态添加属性: {e}")

print("\n💡 何时用 __slots__: 大量小对象 (如数据点、节点、token) + 属性集合固定")


--- __slots__ 属性访问速度 ---
  __dict__  读取 1,000,000 次: 0.191s
  __slots__ 读取 1,000,000 次: 0.150s
  加速比: 1.27x

--- __slots__ 限制 ---
  不能动态添加属性: 'PointSlots' object has no attribute 'w' and no __dict__ for setting new attributes

💡 何时用 __slots__: 大量小对象 (如数据点、节点、token) + 属性集合固定


---
## 4. GIL (Global Interpreter Lock) 深入理解 <a id='4'></a>

### 4.1 GIL 是什么？

**GIL 是 CPython 解释器的一把全局互斥锁**，保证同一时刻只有一个线程执行 Python 字节码。

### 4.2 为什么需要 GIL？

- CPython 的引用计数不是线程安全的
- 没有 GIL → 多线程同时操作引用计数 → 竞态条件 → 内存泄漏/段错误
- GIL 是最简单的解决方案 (以并行性为代价)

### 4.3 GIL 的影响

| 任务类型 | 多线程表现 | 原因 |
|---------|-----------|------|
| CPU 密集 | 更慢！ | GIL 导致线程交替执行 + 上下文切换开销 |
| IO 密集 | 有效加速 | IO 等待时释放 GIL |

### 4.4 GIL 释放的时机

- IO 操作 (文件读写、网络请求)
- C 扩展显式释放 (如 NumPy 的矩阵运算)
- `time.sleep()`
- 每执行约 100 个字节码指令后短暂释放 (Python 3.2+: 每 5ms)

In [ ]:
import threading
import multiprocessing

# 4.1 GIL 影响实验: CPU 密集任务
print("=" * 60)
print("GIL 影响实验: CPU 密集型任务")
print("=" * 60)

def cpu_bound_work(n):
    """纯 CPU 密集任务: 计算素数个数"""
    count = 0
    for i in range(2, n):
        is_prime = True
        for j in range(2, int(i**0.5) + 1):
            if i % j == 0:
                is_prime = False
                break
        if is_prime:
            count += 1
    return count

N = 50000

# 单线程
start = time.perf_counter()
cpu_bound_work(N)
single_thread_time = time.perf_counter() - start
print(f"\n单线程:   {single_thread_time:.3f}s")

# 多线程 (受 GIL 限制)
start = time.perf_counter()
threads = []
for _ in range(2):
    t = threading.Thread(target=cpu_bound_work, args=(N // 2,))
    threads.append(t)
    t.start()
for t in threads:
    t.join()
multi_thread_time = time.perf_counter() - start
print(f"多线程 (2): {multi_thread_time:.3f}s  (比率: {multi_thread_time/single_thread_time:.2f}x)")

# 多进程 (绕过 GIL)
start = time.perf_counter()
with multiprocessing.Pool(2) as pool:
    pool.map(cpu_bound_work, [N // 2, N // 2])
multi_process_time = time.perf_counter() - start
print(f"多进程 (2): {multi_process_time:.3f}s  (比率: {multi_process_time/single_thread_time:.2f}x)")

print(f"\n💡 CPU 密集任务:")
print(f"   多线程反而更慢 (GIL + 上下文切换)")
print(f"   多进程真正并行 (每个进程有自己的 GIL)")

In [ ]:
# 4.2 GIL 释放实验: IO 密集任务
print("\n" + "=" * 60)
print("GIL 释放实验: IO 密集型任务")
print("=" * 60)

def io_bound_work(duration=0.5):
    """模拟 IO 操作 (sleep 时释放 GIL)"""
    time.sleep(duration)

N_TASKS = 8
SLEEP_TIME = 0.5

# 串行
start = time.perf_counter()
for _ in range(N_TASKS):
    io_bound_work(SLEEP_TIME)
serial_time = time.perf_counter() - start
print(f"\n串行 ({N_TASKS} 任务):    {serial_time:.3f}s")

# 多线程
start = time.perf_counter()
threads = [threading.Thread(target=io_bound_work, args=(SLEEP_TIME,)) for _ in range(N_TASKS)]
for t in threads:
    t.start()
for t in threads:
    t.join()
thread_time = time.perf_counter() - start
print(f"多线程 ({N_TASKS} 线程):  {thread_time:.3f}s  (加速: {serial_time/thread_time:.1f}x)")

print(f"\n💡 IO 密集任务: 线程在 sleep/IO 时释放 GIL，其他线程可以执行")
print(f"   → 多线程有效！接近线性加速")

In [ ]:
# 4.3 NumPy 释放 GIL 的例子
import numpy as np

print("\n" + "=" * 60)
print("NumPy 释放 GIL 实验")
print("=" * 60)

def numpy_work(size=2000):
    """NumPy 矩阵乘法 (C 级别释放 GIL)"""
    a = np.random.randn(size, size)
    b = np.random.randn(size, size)
    return np.dot(a, b)

SIZE = 1000

# 单线程 (2次)
start = time.perf_counter()
numpy_work(SIZE)
numpy_work(SIZE)
single_time = time.perf_counter() - start
print(f"\n单线程 (2次 {SIZE}×{SIZE} matmul): {single_time:.3f}s")

# 多线程 (2次并行)
start = time.perf_counter()
t1 = threading.Thread(target=numpy_work, args=(SIZE,))
t2 = threading.Thread(target=numpy_work, args=(SIZE,))
t1.start(); t2.start()
t1.join(); t2.join()
thread_time = time.perf_counter() - start
print(f"多线程 (2并行):               {thread_time:.3f}s  (比率: {thread_time/single_time:.2f}x)")

print(f"\n💡 NumPy 的底层 C/BLAS 代码会释放 GIL")
print(f"   → 多线程可以有效并行 NumPy 计算 (如果 CPU 核心足够)")

---
## 5. threading 多线程 <a id='5'></a>

### 适用场景: IO 密集型任务
- 网络请求、文件读写、数据库查询
- 等待外部资源时释放 GIL

In [ ]:
# 5.1 线程安全问题
print("=" * 60)
print("线程安全: 竞态条件演示")
print("=" * 60)

# 不安全的计数器
counter_unsafe = 0

def increment_unsafe(n):
    global counter_unsafe
    for _ in range(n):
        counter_unsafe += 1  # 读-改-写 不是原子操作！

counter_unsafe = 0
threads = [threading.Thread(target=increment_unsafe, args=(100_000,)) for _ in range(4)]
for t in threads: t.start()
for t in threads: t.join()
print(f"不安全计数器 (期望 400,000): {counter_unsafe:,}  ← 丢失了更新！")

# 安全的计数器 (使用 Lock)
counter_safe = 0
lock = threading.Lock()

def increment_safe(n):
    global counter_safe
    for _ in range(n):
        with lock:
            counter_safe += 1

counter_safe = 0
threads = [threading.Thread(target=increment_safe, args=(100_000,)) for _ in range(4)]
for t in threads: t.start()
for t in threads: t.join()
print(f"安全计数器 (Lock):           {counter_safe:,}  ← 正确！")

In [ ]:
# 5.2 ThreadPoolExecutor
from concurrent.futures import ThreadPoolExecutor, ProcessPoolExecutor, as_completed

print("\n" + "=" * 60)
print("ThreadPoolExecutor 使用示例")
print("=" * 60)

def download_page(url):
    """模拟下载网页"""
    time.sleep(0.3 + 0.2 * (hash(url) % 5) / 5)  # 模拟不同延迟
    return f"Content of {url}" , len(url)

urls = [f"https://example.com/page/{i}" for i in range(10)]

# 串行
start = time.perf_counter()
results_serial = [download_page(url) for url in urls]
serial_time = time.perf_counter() - start

# 线程池
start = time.perf_counter()
with ThreadPoolExecutor(max_workers=5) as executor:
    futures = {executor.submit(download_page, url): url for url in urls}
    results_threaded = []
    for future in as_completed(futures):
        url = futures[future]
        result = future.result()
        results_threaded.append(result)
thread_time = time.perf_counter() - start

print(f"串行: {serial_time:.3f}s")
print(f"线程池 (5 workers): {thread_time:.3f}s  (加速: {serial_time/thread_time:.1f}x)")

---
## 6. multiprocessing 多进程 <a id='6'></a>

### 适用场景: CPU 密集型任务
- 每个进程有独立的 Python 解释器和 GIL
- 真正的并行计算
- 代价: 进程创建开销大、进程间通信 (IPC) 开销

In [ ]:
# 6.1 multiprocessing Pool
print("=" * 60)
print("multiprocessing: CPU 密集型并行")
print("=" * 60)

def compute_sum(args):
    """CPU 密集型: 计算一段范围的和 (含复杂计算)"""
    start, end = args
    total = 0
    for i in range(start, end):
        total += i ** 0.5  # 增加计算量
    return total

TOTAL = 5_000_000
n_workers = os.cpu_count() or 4

# 单进程
start = time.perf_counter()
result_single = compute_sum((0, TOTAL))
single_time = time.perf_counter() - start
print(f"\n单进程: {single_time:.3f}s (result={result_single:.0f})")

# 多进程
chunk_size = TOTAL // n_workers
chunks = [(i * chunk_size, (i + 1) * chunk_size) for i in range(n_workers)]

start = time.perf_counter()
with multiprocessing.Pool(n_workers) as pool:
    results = pool.map(compute_sum, chunks)
result_multi = sum(results)
multi_time = time.perf_counter() - start

print(f"多进程 ({n_workers} workers): {multi_time:.3f}s  (加速: {single_time/multi_time:.2f}x)")
print(f"结果一致: {abs(result_single - result_multi) < 1}")

In [ ]:
# 6.2 进程间通信开销
print("\n" + "=" * 60)
print("进程间通信 (IPC) 开销")
print("=" * 60)

def light_task(x):
    """极轻量任务"""
    return x + 1

def heavy_task(x):
    """重计算任务"""
    total = 0
    for i in range(100_000):
        total += i ** 0.5
    return total + x

data = list(range(100))

# 轻量任务: 多进程反而更慢 (IPC 开销 > 计算)
start = time.perf_counter()
list(map(light_task, data))
serial_light = time.perf_counter() - start

start = time.perf_counter()
with multiprocessing.Pool(4) as pool:
    pool.map(light_task, data)
mp_light = time.perf_counter() - start

print(f"\n轻量任务 (100个):")
print(f"  串行:   {serial_light:.4f}s")
print(f"  多进程: {mp_light:.4f}s  ← 更慢！IPC 开销主导")

# 重计算任务: 多进程有效
start = time.perf_counter()
list(map(heavy_task, data))
serial_heavy = time.perf_counter() - start

start = time.perf_counter()
with multiprocessing.Pool(4) as pool:
    pool.map(heavy_task, data)
mp_heavy = time.perf_counter() - start

print(f"\n重计算任务 (100个):")
print(f"  串行:   {serial_heavy:.3f}s")
print(f"  多进程: {mp_heavy:.3f}s  (加速: {serial_heavy/mp_heavy:.2f}x)")

print("\n💡 经验法则: 当单任务计算时间 >> 进程创建/通信时间时，才值得多进程")

---
## 7. asyncio 异步编程 <a id='7'></a>

### 7.1 核心概念

- **协程 (Coroutine)**: 可暂停/恢复的函数 (`async def`)
- **事件循环 (Event Loop)**: 调度协程的中央调度器
- **await**: 暂停当前协程，让出控制权

### 7.2 vs threading

| | threading | asyncio |
|---|---|---|
| 切换 | OS 调度 (抢占式) | 显式 await (协作式) |
| 开销 | 线程栈 ~8MB | 协程 ~1KB |
| 并发量 | ~1000 线程 | ~100,000 协程 |
| 安全性 | 需要 Lock | 单线程，不需要 Lock |
| 适用 | 通用 IO | 高并发 IO (网络) |

In [ ]:
import asyncio

# 7.1 asyncio 基础
print("=" * 60)
print("asyncio 基础")
print("=" * 60)

async def fetch_data(name, delay):
    """模拟异步 IO 操作"""
    print(f"  [{name}] 开始获取数据...")
    await asyncio.sleep(delay)  # 非阻塞等待
    print(f"  [{name}] 数据获取完成 ({delay}s)")
    return f"{name}_result"

async def main_demo():
    # 串行
    start = time.perf_counter()
    r1 = await fetch_data("A", 1.0)
    r2 = await fetch_data("B", 1.0)
    serial_time = time.perf_counter() - start
    print(f"  串行: {serial_time:.2f}s")
    
    print()
    
    # 并发 (gather)
    start = time.perf_counter()
    r1, r2 = await asyncio.gather(
        fetch_data("C", 1.0),
        fetch_data("D", 1.0)
    )
    concurrent_time = time.perf_counter() - start
    print(f"  并发: {concurrent_time:.2f}s  (加速: {serial_time/concurrent_time:.1f}x)")

# 在 Jupyter 中运行 asyncio
# Jupyter 已经有事件循环，直接 await
try:
    # Jupyter 环境
    await main_demo()
except:
    # 脚本环境
    asyncio.run(main_demo())

In [ ]:
# 7.2 asyncio 高并发演示
print("\n" + "=" * 60)
print("asyncio 高并发性能")
print("=" * 60)

async def quick_io(task_id, delay=0.1):
    """模拟快速 IO"""
    await asyncio.sleep(delay)
    return task_id

async def benchmark_async(n_tasks):
    """异步并发基准测试"""
    start = time.perf_counter()
    tasks = [quick_io(i) for i in range(n_tasks)]
    results = await asyncio.gather(*tasks)
    elapsed = time.perf_counter() - start
    return elapsed, len(results)

async def run_async_benchmark():
    for n in [10, 100, 1000, 5000]:
        elapsed, count = await benchmark_async(n)
        serial_estimate = n * 0.1
        print(f"  {n:5d} 并发任务: {elapsed:.3f}s  "
              f"(串行估计: {serial_estimate:.1f}s, 加速: {serial_estimate/elapsed:.0f}x)")

try:
    await run_async_benchmark()
except:
    asyncio.run(run_async_benchmark())

print("\n💡 asyncio 适合: 大量 IO 密集的并发任务 (如 API 调用、爬虫、WebSocket)")

In [ ]:
# 7.3 Semaphore 控制并发量
print("\n" + "=" * 60)
print("Semaphore: 控制最大并发量")
print("=" * 60)

async def rate_limited_fetch(sem, task_id):
    """受限并发的任务"""
    async with sem:  # 限制同时执行的协程数
        await asyncio.sleep(0.2)
        return task_id

async def semaphore_demo():
    # 限制最多 10 个并发
    sem = asyncio.Semaphore(10)
    
    start = time.perf_counter()
    tasks = [rate_limited_fetch(sem, i) for i in range(50)]
    results = await asyncio.gather(*tasks)
    elapsed = time.perf_counter() - start
    
    print(f"  50 个任务, 最大并发 10: {elapsed:.2f}s")
    print(f"  理论时间: 50/10 × 0.2s = 1.0s")
    print(f"  完成数: {len(results)}")

try:
    await semaphore_demo()
except:
    asyncio.run(semaphore_demo())

---
## 8. 并发模型大 Benchmark <a id='8'></a>

综合对比三种并发模型在不同任务类型下的表现。

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# ===================== CPU 密集型 Benchmark =====================
print("=" * 60)
print("综合 Benchmark: CPU 密集型")
print("=" * 60)

def fib_work(n=35):
    """递归斐波那契 (纯 CPU)"""
    if n <= 1: return n
    return fib_work(n-1) + fib_work(n-2)

N_CPU_TASKS = 4
FIB_N = 32

# 串行
start = time.perf_counter()
for _ in range(N_CPU_TASKS):
    fib_work(FIB_N)
cpu_serial = time.perf_counter() - start

# 多线程
start = time.perf_counter()
threads = [threading.Thread(target=fib_work, args=(FIB_N,)) for _ in range(N_CPU_TASKS)]
for t in threads: t.start()
for t in threads: t.join()
cpu_thread = time.perf_counter() - start

# 多进程
start = time.perf_counter()
with multiprocessing.Pool(N_CPU_TASKS) as pool:
    pool.map(fib_work, [FIB_N] * N_CPU_TASKS)
cpu_process = time.perf_counter() - start

print(f"\nCPU 密集 (fib({FIB_N}) × {N_CPU_TASKS}):")
print(f"  串行:   {cpu_serial:.3f}s")
print(f"  线程:   {cpu_thread:.3f}s")
print(f"  进程:   {cpu_process:.3f}s")

In [ ]:
# ===================== IO 密集型 Benchmark =====================
print("\n" + "=" * 60)
print("综合 Benchmark: IO 密集型")
print("=" * 60)

N_IO_TASKS = 20
IO_DELAY = 0.2

# 串行
start = time.perf_counter()
for _ in range(N_IO_TASKS):
    time.sleep(IO_DELAY)
io_serial = time.perf_counter() - start

# 多线程
start = time.perf_counter()
with ThreadPoolExecutor(max_workers=N_IO_TASKS) as executor:
    list(executor.map(time.sleep, [IO_DELAY] * N_IO_TASKS))
io_thread = time.perf_counter() - start

# asyncio
async def io_async_benchmark():
    tasks = [asyncio.sleep(IO_DELAY) for _ in range(N_IO_TASKS)]
    await asyncio.gather(*tasks)

start = time.perf_counter()
try:
    await io_async_benchmark()
except:
    asyncio.run(io_async_benchmark())
io_async = time.perf_counter() - start

print(f"\nIO 密集 ({N_IO_TASKS} 任务, 每个 {IO_DELAY}s):")
print(f"  串行:   {io_serial:.3f}s")
print(f"  线程:   {io_thread:.3f}s  (加速: {io_serial/io_thread:.1f}x)")
print(f"  asyncio: {io_async:.3f}s  (加速: {io_serial/io_async:.1f}x)")

In [ ]:
# ===================== 综合可视化 =====================
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# CPU 密集型
methods = ['Serial', 'Threading', 'Multiprocessing']
cpu_times = [cpu_serial, cpu_thread, cpu_process]
colors = ['#78909C', '#EF5350', '#4CAF50']
bars = axes[0].bar(methods, cpu_times, color=colors, width=0.5)
axes[0].set_ylabel('Time (s)')
axes[0].set_title(f'CPU 密集型: fib({FIB_N}) × {N_CPU_TASKS}')
for bar, t in zip(bars, cpu_times):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02, 
                f'{t:.3f}s', ha='center', va='bottom', fontweight='bold')

# IO 密集型
methods_io = ['Serial', 'Threading', 'Asyncio']
io_times = [io_serial, io_thread, io_async]
colors_io = ['#78909C', '#EF5350', '#2196F3']
bars = axes[1].bar(methods_io, io_times, color=colors_io, width=0.5)
axes[1].set_ylabel('Time (s)')
axes[1].set_title(f'IO 密集型: sleep({IO_DELAY}) × {N_IO_TASKS}')
for bar, t in zip(bars, io_times):
    axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02, 
                f'{t:.3f}s', ha='center', va='bottom', fontweight='bold')

plt.suptitle('Python 并发模型对比', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

# 总结表格
print("\n" + "=" * 70)
print("选型指南")
print("=" * 70)
print(f"{'场景':<20s} | {'推荐方案':<20s} | {'原因':<30s}")
print("-" * 70)
print(f"{'CPU 密集计算':<20s} | {'multiprocessing':<20s} | {'绕过 GIL，真正并行':<30s}")
print(f"{'IO 密集 (少量)':<20s} | {'threading':<20s} | {'简单，等待时释放 GIL':<30s}")
print(f"{'IO 密集 (大量)':<20s} | {'asyncio':<20s} | {'协程轻量，万级并发':<30s}")
print(f"{'混合型':<20s} | {'asyncio + Process':<20s} | {'IO 用 async，CPU 用进程池':<30s}")

---
## 9. asyncio 爬虫 Demo <a id='9'></a>

一个结构化的异步爬虫框架，展示 asyncio 在实际场景中的应用。

> 注意: 这里用模拟的 HTTP 请求 (不需要网络)，结构与真实爬虫一致。

In [ ]:
import random
from dataclasses import dataclass, field
from typing import List, Optional

@dataclass
class CrawlResult:
    url: str
    status: int
    content_length: int
    elapsed: float


class AsyncCrawler:
    """
    异步爬虫框架
    
    特性:
    - Semaphore 控制并发量
    - 指数退避重试
    - 结果收集与统计
    """
    def __init__(self, max_concurrent=10, max_retries=3):
        self.semaphore = asyncio.Semaphore(max_concurrent)
        self.max_retries = max_retries
        self.results: List[CrawlResult] = []
        self.errors: List[dict] = []
    
    async def fetch(self, url: str) -> CrawlResult:
        """
        模拟 HTTP 请求
        (实际项目中替换为 aiohttp.ClientSession.get(url))
        """
        # 模拟网络延迟
        delay = random.uniform(0.05, 0.3)
        await asyncio.sleep(delay)
        
        # 模拟偶尔失败
        if random.random() < 0.1:  # 10% 失败率
            raise ConnectionError(f"Failed to fetch {url}")
        
        # 模拟返回数据
        content_length = random.randint(1000, 50000)
        return CrawlResult(
            url=url,
            status=200,
            content_length=content_length,
            elapsed=delay
        )
    
    async def fetch_with_retry(self, url: str) -> Optional[CrawlResult]:
        """带重试的请求"""
        async with self.semaphore:
            for attempt in range(self.max_retries):
                try:
                    result = await self.fetch(url)
                    return result
                except Exception as e:
                    if attempt < self.max_retries - 1:
                        wait = 2 ** attempt * 0.1  # 指数退避
                        await asyncio.sleep(wait)
                    else:
                        self.errors.append({'url': url, 'error': str(e)})
                        return None
    
    async def crawl(self, urls: List[str]) -> List[CrawlResult]:
        """并发爬取所有 URL"""
        tasks = [self.fetch_with_retry(url) for url in urls]
        results = await asyncio.gather(*tasks)
        self.results = [r for r in results if r is not None]
        return self.results
    
    def print_stats(self):
        """打印统计信息"""
        if not self.results:
            print("  无结果")
            return
        
        total = len(self.results) + len(self.errors)
        elapsed_times = [r.elapsed for r in self.results]
        sizes = [r.content_length for r in self.results]
        
        print(f"  总请求数: {total}")
        print(f"  成功: {len(self.results)} ({len(self.results)/total*100:.0f}%)")
        print(f"  失败: {len(self.errors)} ({len(self.errors)/total*100:.0f}%)")
        print(f"  平均延迟: {np.mean(elapsed_times):.3f}s")
        print(f"  总数据量: {sum(sizes)/1024:.1f} KB")


# 运行爬虫
print("=" * 60)
print("asyncio 爬虫 Demo")
print("=" * 60)

urls = [f"https://example.com/api/data/{i}" for i in range(100)]

async def run_crawler():
    crawler = AsyncCrawler(max_concurrent=20, max_retries=3)
    
    start = time.perf_counter()
    results = await crawler.crawl(urls)
    elapsed = time.perf_counter() - start
    
    print(f"\n  完成时间: {elapsed:.2f}s")
    print(f"  串行估计: ~{len(urls) * 0.15:.1f}s")
    print(f"  加速比:   ~{len(urls) * 0.15 / elapsed:.0f}x")
    print()
    crawler.print_stats()

try:
    await run_crawler()
except:
    asyncio.run(run_crawler())

---
## 10. 思考题 & 总结 <a id='10'></a>

### 今日核心知识点

1. **对象模型**: 一切皆对象，变量是引用 (标签) 而非盒子
2. **引用计数 + GC 三代**: 主力回收 + 循环引用处理
3. **`__slots__`**: 固定属性 → 省内存 + 略快，适合大量小对象
4. **GIL**: CPU 密集用多进程，IO 密集用多线程/asyncio
5. **三种并发模型**: threading (简单 IO) / multiprocessing (CPU) / asyncio (高并发 IO)

### 面试高频问题

1. **Python 的内存管理机制是什么？** → 引用计数 + 分代 GC
2. **什么是 GIL？对多线程有什么影响？** → 全局锁，CPU 密集无法并行
3. **如何解决 GIL 的限制？** → multiprocessing / C 扩展 / 其他 Python 实现
4. **asyncio 和 threading 的区别？** → 协作式 vs 抢占式
5. **什么时候用 `__slots__`？** → 大量实例、固定属性、内存敏感
6. **循环引用怎么处理？** → GC 检测 / weakref 预防

### 明日预习: Python 元编程 + 设计模式
- 装饰器的三层结构
- 描述符协议 (`__get__`, `__set__`)
- 元类 (`__new__` vs `__init__`)

In [ ]:
print("\n" + "=" * 60)
print("W1 Day1 完成! 🎉")
print("=" * 60)
print("""
今日成果:
  ✅ Python 对象模型 (id/type/value, 引用语义, 小整数缓存)
  ✅ 引用计数实验 + 循环引用 + GC 三代晋升
  ✅ weakref 弱引用
  ✅ __slots__ 内存优化 (100K 实例对比)
  ✅ GIL 深入: CPU 密集 vs IO 密集 vs NumPy 释放 GIL
  ✅ threading 线程安全 + Lock + ThreadPoolExecutor
  ✅ multiprocessing Pool + IPC 开销分析
  ✅ asyncio 协程 + gather + Semaphore
  ✅ 综合 Benchmark (CPU/IO × 串行/线程/进程/async)
  ✅ asyncio 爬虫 Demo (重试 + 并发控制 + 统计)
""")